# Blok 2 · Zadanie grupowe: kto nie spłaci kredytu? 💳

**Sytuacja:** jesteście zespołem data science w banku. Dział ryzyka chce modelu, który dla każdego klienta oszacuje **prawdopodobieństwo poważnych problemów ze spłatą w ciągu 2 lat** (*probability of default*, PD). Na podstawie PD bank decyduje o przyznaniu kredytu i liczy rezerwy na straty — więc liczą się nie tylko „kto jest ryzykowny”, ale też **czy p jest wiarygodne**.

Target `SeriousDlqin2yrs` (1 = opóźnienie 90+ dni w ciągu 2 lat) → **klasyfikacja binarna.

## Zasady

* Grupy 2–3 osobowe, macie **ok. 50 minut**.
* Przechodzicie przez kroki 0–8. Komórki z `# TODO` uzupełniacie sami.
* **Każdą decyzję potwierdzacie wynikiem CV** i zapisujecie w **dzienniku eksperymentów**
* Narzędzia znacie z dzisiejszych notebooków `01_demo_ocena_modeli` i `02_demo_lepsze_dane` — korzystajcie z nich śmiało.

## Dane — 10 cech + target

Dane z konkursu Kaggle *Give Me Some Credit* (150 000 kredytobiorców), pobierane z OpenML.

| kolumna | znaczenie |
|---|---|
| `SeriousDlqin2yrs` | **TARGET**: 1 = opóźnienie 90+ dni w ciągu 2 lat |
| `RevolvingUtilizationOfUnsecuredLines` | wykorzystanie limitów kart kredytowych i linii (saldo / limit) |
| `age` | wiek kredytobiorcy |
| `NumberOfTime30-59DaysPastDueNotWorse` | ile razy w ciągu 2 lat spóźnił się 30–59 dni |
| `NumberOfTime60-89DaysPastDueNotWorse` | ile razy spóźnił się 60–89 dni |
| `NumberOfTimes90DaysLate` | ile razy spóźnił się 90+ dni |
| `DebtRatio` | miesięczne zobowiązania / miesięczny dochód |
| `MonthlyIncome` | miesięczny dochód |
| `NumberOfOpenCreditLinesAndLoans` | liczba otwartych kredytów i linii kredytowych |
| `NumberRealEstateLoansOrLines` | liczba kredytów hipotecznych |
| `NumberOfDependents` | liczba osób na utrzymaniu |

## Krok 0: Wczytanie danych i próbka (~3 min)

Pełny zbiór ma 150 000 wierszy. Żeby eksperymenty liczyły się szybko, pracujemy na **losowej próbce 30 000** (stratyfikowanej — z tym samym odsetkiem defaultów). Komórka jest gotowa — uruchomcie ją.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

TARGET = "SeriousDlqin2yrs"

def na_liczbe(s):
    # target z OpenML przychodzi jako tekst/bajty '0'/'1' — zamieniamy na int
    return s.map(lambda v: v.decode() if isinstance(v, bytes) else v).astype(int)

try:
    df_pelny = fetch_openml(data_id=45577, as_frame=True).frame
except Exception:
    # zapasowo: plik ARFF pobrany ręcznie z https://www.openml.org/d/45577 do folderu data/
    from scipy.io import arff
    df_pelny = pd.DataFrame(arff.loadarff("data/give-me-some-credit")[0])
df_pelny[TARGET] = na_liczbe(df_pelny[TARGET])

df, _ = train_test_split(df_pelny, train_size=30_000, stratify=df_pelny[TARGET], random_state=RANDOM_STATE)
df = df.reset_index(drop=True)
X = df.drop(columns=TARGET)
y = df[TARGET]
print("pełny zbiór:", df_pelny.shape, "  próbka:", df.shape)
df.head()

## Krok 1: EDA pod kątem dzisiejszych zajęć (~5 min)

Odpowiedzcie na cztery pytania:

a. **Jaki odsetek klientów ma default?** Co to oznacza dla wyboru metryki?

b. **Które kolumny mają braki i ile procent?**

c. **Które kolumny mają wartości odstające?**

d. **Czy są wartości podejrzane / niemożliwe?**

In [ ]:
# TODO 1a

# TODO 1b

# TODO 1c

# TODO 1d

## Krok 2: Wybór metryki (~3 min)

Na podstawie analizy powyżej wybierzcie jaką metryką chcecie się kierować i dlaczego:
* jaka główna?
* jaka inna pomocniczna?

## Krok 3: Baseline + CV + dziennik eksperymentów (~5 min)

Poniżej gotowe narzędzia:
* jeden obiekt `cv` na całe zadanie — **wszystkie warianty oceniamy na tych samych foldach**,
* funkcja `zapisz(nazwa, model, uwagi)` liczy CV i dopisuje wiersz do dziennika.

Waszym zadaniem jest dodać dwa pierwsze wiersze: **Dummy** oraz **regresję logistyczną** z najprostszym preprocessingiem oraz wybrane metryki.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
dziennik = []

def zapisz(nazwa, model, uwagi=""):
    r = cross_validate(model, X, y, cv=cv, scoring=["average_precision", "roc_auc", "neg_brier_score"])
    dziennik.append({"wariant": nazwa,
                    #TODO: metryki np.
                    # "PR-AUC": r["test_average_precision"].mean(),
                     "uwagi": uwagi})
    return pd.DataFrame(dziennik).set_index("wariant").round(4)

def LR(**kwargs):
    return LogisticRegression(max_iter=3000, **kwargs)

In [ ]:
# TODO 3: Dummy i regresja logistyczna z najprosztszym preprocessingiem
# zapisz("Dummy", ..., "punkt odniesienia")
# zapisz("baseline: regresja logistyczna", make_pipeline(...), "")

## Krok 4: Wartości odstające i dziwne kody (~10 min)

W kroku 1 znaleźliście wartości podejrzane. Teraz je obsłużcie — **wszystko wewnątrz pipeline'u**.

**4a. Kody → brak.** Uzupełnijcie funkcję, która w kolumnach z kodow zamieni je na `NaN`. Dzięki `FunctionTransformer` stanie się krokiem pipeline'u, a imputer z `add_indicator=True` doda wskaźnik „tu był kod”.

**4b. Skrajne ogony.** Wypróbujcie co najmniej dwa sposoby: winsoryzację (klasa gotowa poniżej — ta sama co w demo) oraz `log1p`. Możecie też połączyć kroki.

Każdy wariant → nowy wiersz w dzienniku.

In [2]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

class Winsoryzacja(BaseEstimator, TransformerMixin):
    # przycina każdą kolumnę do kwantyli [q, 1-q] wyznaczonych na danych treningowych (jak w demo)
    def __init__(self, q=0.01):
        self.q = q
    def fit(self, X, y=None):
        A = np.asarray(X, dtype=float)
        self.dolne_ = np.nanquantile(A, self.q, axis=0)
        self.gorne_ = np.nanquantile(A, 1 - self.q, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.dolne_, self.gorne_)

In [ ]:
# KOLUMNY_Z_KODAMI = [...]

def kody_na_braki(X):
    X = X.copy()
    # TODO 4a
    return X

zamiana_kodow = FunctionTransformer(kody_na_braki)

# TODO 4a: zapisz(...) — pipeline

# TODO 4b: winsoryzacja, log1p, połączenie z 4a — każdy wariant jako osobny wiersz dziennika

💬 **Do dyskusji:** o ile poprawił się wynik? Porównajcie to z różnicami, które widzieliśmy w demo — czy to szum, czy realna poprawa?

## Krok 5: Braki (~8 min)

Kolumna z największą liczbą braków (krok 1b) ma ok. 20% braków. Na pipeline'ie z kroku 4 porównajcie imputery:
* mediana,
* mediana + wskaźnik braku,
* `IterativeImputer`,
* `KNNImputer` (pamiętajcie o skalowaniu **przed** nim; liczy się ok. 30 s).

💬 **Do dyskusji:** czy brak dochodu sam w sobie mówi coś o ryzyku? Sprawdźcie odsetek defaultów wśród klientów z brakiem i bez braku dochodu.

In [ ]:
from sklearn.experimental import enable_iterative_imputer   
from sklearn.impute import IterativeImputer, KNNImputer

def z_kodami_i_winsoryzacja(imputer):
    return make_pipeline(zamiana_kodow, imputer, Winsoryzacja(0.01), StandardScaler(), LR())

# TODO 5: porównajcie imputery (zapisz w dzienniku)

# TODO 5 (dyskusja): odsetek defaultów przy braku vs bez braku MonthlyIncome

## Krok 6: Selekcja cech (~5 min)

Po preprocessingu (z kolumnami-wskaźnikami) model ma ok. 15 cech. Sprawdźcie:
* `SelectFromModel` z regresją logistyczną L1 (`penalty="l1", solver="liblinear"`, np. `C=0.01`) — ile cech zostaje i co się dzieje z wynikiem?
* **albo** `RFECV` (`scoring="average_precision"`).

Selekcja to krok pipeline'u — wstawcie ją **przed** klasyfikatorem.

In [ ]:
from sklearn.feature_selection import SelectFromModel, RFECV

# TODO 6: pipeline z selekcją, zapis w dzienniku; wypiszcie, ile cech zostało

## Krok 7: Kalibracja (~7 min)

Bank używa wartości p wprost, więc sprawdźcie, czy są wiarygodne. Na najlepszym pipeline'ie porównajcie:
1. regresję logistyczną **bez wag**,
2. z `class_weight="balanced"`,
3. wariant 2 opakowany w `CalibratedClassifierCV(..., method="sigmoid", cv=5)`.

Dla każdego: PR-AUC, Brier (z dziennika), **średnie przewidywane p** (porównajcie z odsetkiem defaultów) i krzywa kalibracji (`CalibrationDisplay.from_predictions` na predykcjach z `cross_val_predict`).

💬 **Do dyskusji:** którego modelu bank powinien użyć do wyliczania PD — i dlaczego?

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.model_selection import cross_val_predict

# TODO 7: trzy warianty -> dziennik, średnie p, krzywe kalibracji na jednym wykresie

## Krok 8: Finał — dziennik i wnioski (~4 min)

1. Wyświetlcie cały dziennik, posortowany po PR-AUC.
2. Wybierzcie **jeden** model, który polecacie bankowi, i uzasadnijcie wybór w 3 zdaniach: (a) który krok dał najwięcej, (b) co **nie** pomogło, (c) dlaczego ten model nadaje się do PD.
3. Sprawdźcie, że Wasz model przyjmuje **surowe dane**.

In [ ]:
# TODO 8.1: dziennik posortowany po PR-AUC

# TODO 8.3: dopasujcie wybrany model na całej próbce i sprawdźcie predykcję na 5 surowych wierszach
# model_finalny = ...
# model_finalny.fit(X, y)
# print(model_finalny.predict_proba(df_pelny.drop(columns=TARGET).sample(5, random_state=0))[:, 1])

✍️ **Wasze wnioski (3 zdania):**

*(a) najwięcej dało…*

*(b) nie pomogło…*

*(c) do PD polecamy…, bo…*

## (Opcjonalnie) Krok 9: Zweryfikuj czy na wszystkich danych (150k zamiast 30k) wnioski są podobne?